# Start the virtual try-on API

Press **Run All** (⏩). Nothing else to do.

It starts the API, reconnects the fixed address `https://tryon.example.com`,
and warms the generator so your first real image is fast rather than taking
a minute and a half.

About 3 minutes normally. If the container came back empty it rebuilds
itself first, which takes about 10.

In [ ]:
# API + named tunnel. Streams the log so you can see where it is.
import subprocess, sys

p = subprocess.Popen(
    ['bash', '/workspace/tryon/bootstrap.sh'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
    sys.stdout.flush()
p.wait()

In [ ]:
# Warm the generator so the first real image is fast.
#
# The first generation after any restart loads 18 GB of weights from disk -
# a 9.7 GB UNet and an 8.3 GB text encoder - which takes 40-90s. Every one
# after that is about 6s. Doing one throwaway image here means that cost is
# paid now, while you are still opening the page, instead of on your first
# real try-on where it looks like the page has hung.
import time, json, urllib.request, glob, os

API = 'http://127.0.0.1:8000'
ROOT = '/workspace/tryon'

person = sorted(glob.glob(f'{ROOT}/inputs/models/*.jpeg'))
garment = sorted(glob.glob(f'{ROOT}/inputs/fg/*.jpeg'))
if not (person and garment):
    print('no sample images on the box; skipping warm-up')
else:
    import subprocess
    t0 = time.time()
    print('warming the generator (40-90s, one time)...', flush=True)
    out = subprocess.run([
        'curl', '-s', '-m', '600', '-X', 'POST',
        f'{API}/v1/tryon?wait=true&wait_timeout=580',
        '-F', f'person=@{person[0]}', '-F', f'garment=@{garment[0]}',
        '-F', 'guardrail=false', '-F', 'megapixels=0.5',
    ], capture_output=True, text=True).stdout
    try:
        d = json.loads(out)
        print(f"warm-up {d.get('status')} in {d.get('duration_seconds')}s")
    except Exception:
        print('warm-up did not report cleanly:', out[:200])
    print(f'total {time.time()-t0:.0f}s — the next image will be about 6s')


In [ ]:
# Confirm the public address is answering, not just the local one.
import urllib.request, json

try:
    with urllib.request.urlopen('https://tryon.example.com/healthz', timeout=30) as r:
        print('API status', r.status)
    with urllib.request.urlopen('https://tryon.example.com/v1/garments', timeout=30) as r:
        d = json.load(r)
        described = len([g for g in d['garments'] if g.get('pieces')])
        print(f"library: {d['count']} garments, {described} described")
    print()
    print('READY — open the page:')
    print('https://<your-bucket>.s3.amazonaws.com/demo/index.html')
    print('login is already filled in')
except Exception as e:
    print('not reachable yet:', e)
    print('wait a few seconds and re-run this cell')
